In [ ]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from langchain_core.messages import BaseMessage,HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph.message import add_messages
from dotenv import load_dotenv

from langgraph.prebuilt import ToolNode,tools_condition
from langchain_tavily import TavilySearch
from langchain.tools import tool

import requests
import random
import os


In [ ]:
load_dotenv()

In [ ]:
llm=ChatOpenAI(model='gpt-4o-mini')

In [ ]:
search_tool=TavilySearch(max_results=3)

In [ ]:
@tool
def calculator(first_num: float, second_num: float, operation: str) -> dict:
    """
    Perform a basic arithmetic operation on two numbers.
    Supported operations: add, sub, mul, div
    """
    try:
        if operation == "add":
            result = first_num + second_num
        elif operation == "sub":
            result = first_num - second_num
        elif operation == "mul":
            result = first_num * second_num
        elif operation == "div":
            if second_num == 0:
                return {"error": "Division by zero is not allowed"}
            result = first_num / second_num
        else:
            return {"error": f"Unsupported operation '{operation}'"}
        
        return {"first_num": first_num, "second_num": second_num, "operation": operation, "result": result}
    except Exception as e:
        return {"error": str(e)}


@tool
def get_stock_price(symbol: str) -> dict: #type:ignore
    """
    Fetch latest stock price for a given symbol (e.g. 'AAPL', 'TSLA') 
    using Alpha Vantage with API key in the URL.
    """
    try:
        api_key=os.getenv('ALPHAVANTAGE_API_KEY')
    except:
        print("ALPHAVANTAGE_API_KEY not found")
      
    try:  
        url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey={api_key}"
        r = requests.get(url)
        return r.json()
        
    except Exception as e:
        return {"error": str(e)}

In [ ]:
tools=[search_tool,calculator,get_stock_price]

In [ ]:
llm_with_tools=llm.bind_tools(tools)

In [ ]:
class ChatState(TypedDict):
    messages:Annotated[list[str],add_messages]

In [ ]:
# graph nodes
def chat_node(state: ChatState):
    """LLM node that may answer or request a tool call."""
    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

tool_node = ToolNode(tools)  # Executes tool calls

In [ ]:
# graph structure
graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)

In [ ]:
graph.add_edge(START, "chat_node")

# If the LLM asked for a tool, go to ToolNode; else finish
graph.add_conditional_edges("chat_node", tools_condition)

graph.add_edge("tools", 'chat_node')    

In [ ]:
chatbot = graph.compile()

In [ ]:
chatbot

In [ ]:
out = chatbot.invoke({"messages": [HumanMessage(content="Hello!")]}) #type:ignore

print(out["messages"][-1].content)

In [ ]:
# Chat requiring tool
out = chatbot.invoke({"messages": [HumanMessage(content="What is 2*3?")]}) #type:ignore
print(out["messages"][-1].content)

In [ ]:
out = chatbot.invoke({"messages": [HumanMessage(content="What is the stock price of apple")]}) #type:ignore
print(out["messages"][-1].content)

In [ ]:
# Chat requiring tool
out = chatbot.invoke({"messages": [HumanMessage(content="First find out the stock price of Apple using get stock price tool then use the calculator tool to find out how much will it take to purchase 50 shares?")]}) #type:ignore
print(out["messages"][-1].content)

In [ ]:
# Chat requiring tool
out = chatbot.invoke({"messages": [HumanMessage(content="First find out the stock price of Apple using get stock price tool then use the calculator tool to find out how much will it take to purchase 50 shares? and then find out the capital city of cape verde")]}) #type:ignore
print(out["messages"][-1].content)